# The central five-qutrit comparison

Variational $((5,3,3))_3$ code (40 MS gates) versus the $[[5,1,3]]_{Z_3}$ stabilizer
code (36 MS gates) at $n=5$, $q=3$ — paper Sec. VI A, Table I, App. B/C/F.

This notebook recomputes, from the committed parameters and the `src/` library only:

1. **Setup** — calibrated constants and the strength grid.
2. **The two codes** — codewords, Knill–Laflamme residuals, weight enumerators (Table II).
3. **Compilation** — the native-gate ansatz (Table III row) and the 36-MS encoder of App. C,
   with one CZ → 4 MS decomposition spelled out.
4. **Noise model** — one calibrated gate step of Eq. (8) from the App. B Kraus families.
5. **Recovery calculation** — Table I: projective syndrome decoder (App. F), Petz recovery
   (Eq. F2), certified optimal recovery.
6. **i.i.d. certification benchmark** — lookup-decoder LER at $p = 0.05$ and the
   weight-conditioned probes.
7. **Summary** — paper value vs notebook value for everything above.

Run from the repository root with `PYTHONPATH=.`, or execute in place with
`jupyter nbconvert --to notebook --execute --inplace notebooks/central_comparison.ipynb`.
Nothing is retrained; every number comes from `results/` and the code in `src/`.

## 1. Setup

In [1]:
import os
import sys
import time
import math

T_START = time.time()

# the notebook lives in notebooks/; make the repository root importable
REPO = os.path.abspath(os.path.join(os.getcwd(), ".."))
if not os.path.isdir(os.path.join(REPO, "src")):
    REPO = os.getcwd()
sys.path.insert(0, REPO)

import numpy as np
import src.jax_backend  # noqa: F401  -- enables 64-bit JAX before any encoder is built
from src import central_comparison as cc

# ---- knobs ---------------------------------------------------------------
N_SHOTS = 2_000_000       # i.i.d. benchmark shots per code (paper: 2e6; use 2e5 for a quick run)
SEED_IID = 3000           # seed of the paper's central-comparison measurement
S_OPTIMAL = cc.S_GRID     # strengths at which the certified optimal recovery is computed
# ---------------------------------------------------------------------------

print(f"sigma_p^2 = {cc.SIGMA_P2}  ->  eta = e^(-sigma_p^2) = {cc.ETA_CAL:.4f}")
print(f"sigma_a^2 = {cc.SIGMA_A2}")
print("strength grid s =", cc.S_GRID, "(s = 1 is the calibrated point)")
print("eta(s)       =", [round(cc.eta_of_s(s), 4) for s in cc.S_GRID])
print("sigma_a^2(s) =", [round(cc.sigma_a2_of_s(s), 4) for s in cc.S_GRID])

SUMMARY = []   # (quantity, paper value, notebook value) rows for Sec. 7

def record(name, paper, value, fmt="{:.4g}"):
    SUMMARY.append((name, paper, value, fmt))

sigma_p^2 = 0.073  ->  eta = e^(-sigma_p^2) = 0.9296
sigma_a^2 = 0.047
strength grid s = (0.125, 0.25, 0.5, 1.0, 1.5) (s = 1 is the calibrated point)
eta(s)       = [0.9909, 0.9819, 0.9642, 0.9296, 0.8963]
sigma_a^2(s) = [0.0059, 0.0118, 0.0235, 0.047, 0.0705]


## 2. The two codes

* The **central code** is the selected run of the channel-weighted protocol at $L=2$
  (paper Sec. VI A, step iii); its parameters are the committed file behind the catalog
  entry `central_553_3`. The codewords are rebuilt by running the ansatz of Sec. IV A.
* The **stabilizer code** $[[5,1,3]]_{Z_3}$ is written down from Eq. (13): amplitude
  $q^{-2}\,\omega^{\sum_i k_i k_{i+1}}$ on every basis state with $\sum_i k_i \equiv a$.

In [2]:
from src.catalog import load_code, code_path

central = load_code("central_553_3")
CS_CEN = np.asarray(central["code_states"])          # (3, 243)
CS_STAB = cc.stabilizer_code_states()                 # (3, 243)

print("central code parameter file :", os.path.relpath(code_path("central_553_3"), REPO))
print("layers L =", central["metadata"]["n_layers"],
      "| parameter array shape", tuple(central["params"].shape),
      "| connectivity edges", len(central["metadata"]["connections"]))
for label, cs in (("central", CS_CEN), ("stabilizer", CS_STAB)):
    gram = cs @ cs.conj().T
    print(f"{label:10s} codewords {cs.shape}, orthonormality deviation "
          f"{np.max(np.abs(gram - np.eye(3))):.1e}")

central code parameter file : results/campaigns/r1_campaign/t1_codes/A_L2_s100.npz
layers L = 2 | parameter array shape (2, 90) | connectivity edges 10
central    codewords (3, 243), orthonormality deviation 4.4e-16
stabilizer codewords (3, 243), orthonormality deviation 7.8e-16


**Knill–Laflamme residuals.** For the correction set $\{I\} \cup \{\text{weight-1 errors}\}$
of the channel error basis (Sec. IV C) we evaluate every product $E_a^\dagger E_b$ and report
the worst off-diagonal element $|\langle\psi_i|E_a^\dagger E_b|\psi_j\rangle|$, $i\neq j$, and the
worst variance of the diagonal elements over $k$. The stabilizer code satisfies the
conditions exactly; the variational code is an *approximate* distance-3 code.

**Detection loss, Eq. (12)** on the completed detection set: the paper quotes the
"residual" of the central code on the full basis (Table IV: 4.6) — the loss that the
unweighted protocol variants minimise — and the channel-weighted protocol's own
completed channel basis.

In [3]:
import jax.numpy as jnp
from src.errors import ErrorModel, qudit_hardware_error_basis
from src.analysis import compute_kl_residuals
from src.jax_backend import create_jax_loss_vmap

E_det_dense, E_corr_dense = ErrorModel(d=3, n_qudit=5, distance=3, closed=True).build_dense()
print(f"completed channel basis at n=5: |E_det| = {len(E_det_dense)} (incl. identity), "
      f"|E_corr| = {len(E_corr_dense)} (identity + weight-1)")
for label, cs in (("central", CS_CEN), ("stabilizer", CS_STAB)):
    off, var = compute_kl_residuals(cs, E_corr_dense)
    print(f"{label:10s} max |<i|E_a^+E_b|j>| (i!=j) = {off:.3e}   max Var_k <k|E_a^+E_b|k> = {var:.3e}")

losses = {}
for basis, label in ((None, "completed channel basis"), ("full", "full basis (Table IV 'residual')")):
    grouped = ErrorModel(d=3, n_qudit=5, distance=3, closed=True, basis=basis).build_grouped(verbose=False)
    n_ops = sum(int(g["matrices"].shape[0]) for g in grouped if len(g["wires"]) > 0)
    loss_fn = create_jax_loss_vmap(grouped, K=3, d=3, n_qudit=5, distance=3)
    for code, cs in (("central", CS_CEN), ("stabilizer", CS_STAB)):
        losses[(basis, code)] = float(loss_fn(jnp.asarray(cs)))
    print(f"{label:38s} ({n_ops:3d} operators): loss central = {losses[(basis,'central')]:.4f}, "
          f"stabilizer = {losses[(basis,'stabilizer')]:.1e}")
record("Table IV residual, central (full-basis loss)", 4.6, losses[("full", "central")], "{:.2f}")
record("Table IV residual, stabilizer", "~1e-30", losses[("full", "stabilizer")], "{:.1e}")

completed channel basis at n=5: |E_det| = 221 (incl. identity), |E_corr| = 21 (identity + weight-1)


central    max |<i|E_a^+E_b|j>| (i!=j) = 8.646e-02   max Var_k <k|E_a^+E_b|k> = 7.804e-03


stabilizer max |<i|E_a^+E_b|j>| (i!=j) = 9.086e-17   max Var_k <k|E_a^+E_b|k> = 8.217e-33


completed channel basis                (220 operators): loss central = 0.9888, stabilizer = 8.9e-31


full basis (Table IV 'residual')       (800 operators): loss central = 4.5994, stabilizer = 2.6e-30


The 800 non-identity operators of the full basis above are the set the manuscript's residuals were computed over; the completed full set as defined in Sec. IV C has 1190 (the wire-grouped builder drops zero-product pairs of the non-unitary lowering/raising operators). Over the complete set the central code's residual is 5.81 instead of 4.60. See `docs/HANDOFF.md`, item A1.

**Shor–Laflamme weight enumerators (Table II).** $A_j = K^{-2}\sum_{\mathrm{wt}(E)=j}|\mathrm{Tr}(PE)|^2$,
$B_j = K^{-1}\sum_{\mathrm{wt}(E)=j}\mathrm{Tr}(PEPE^\dagger)$ over products of the single-qudit
channel error basis $\{Z_1, Z_2, X_{01}, X_{12}\}$; $B_j - A_j = 0$ for $j<d$ is the exact-detection
condition. Expected rows: central 2.4408 / 2.4229 / 2.4631 / 3.1225, stabilizer 2.2222 / 1.9753 with zero deficit.

In [4]:
from src.analysis import compute_weight_enumerators

basis = [np.asarray(E) for E in qudit_hardware_error_basis(3)]
enum = {}
print(f"{'code':10s} {'A1':>8s} {'A2':>8s} {'B1':>8s} {'B2':>8s} {'B1-A1':>8s} {'B2-A2':>8s}")
for label, cs in (("central", CS_CEN), ("stabilizer", CS_STAB)):
    A, B = compute_weight_enumerators(cs, 5, 3, basis, max_weight=2)
    enum[label] = (A, B)
    print(f"{label:10s} {A[1]:8.4f} {A[2]:8.4f} {B[1]:8.4f} {B[2]:8.4f} {B[1]-A[1]:8.4f} {B[2]-A[2]:8.4f}")
A, B = enum["central"]
record("Table II central A1", 2.4408, A[1]); record("Table II central A2", 2.4229, A[2])
record("Table II central B1", 2.4631, B[1]); record("Table II central B2", 3.1225, B[2])
A, B = enum["stabilizer"]
record("Table II stabilizer A1 (=B1)", 2.2222, A[1]); record("Table II stabilizer A2 (=B2)", 1.9753, A[2])
record("Table II stabilizer B2-A2", 0.0, B[2] - A[2], "{:.1e}")

code             A1       A2       B1       B2    B1-A1    B2-A2


central      2.4408   2.4229   2.4631   3.1225   0.0223   0.6996


stabilizer   2.2222   1.9753   2.2222   1.9753  -0.0000   0.0000


## 3. Compilation

### (a) The variational encoder: one ansatz layer (Fig. 2)

Each layer applies, in this order, two rounds of XY rotations on every qudit (one per
adjacent transition), one MS gate per (edge, transition) of the all-to-all graph, and one
Z phase per (qudit, transition). At $n=5$, $q=3$: $|E| = 10$, $P_\ell = (5n + 2|E|)(q-1) = 90$,
180 parameters for $L = 2$, and the MS budget $L|E|(q-1) = 40$ (Table III).

In [5]:
from src.encoder import layer_gate_sequence

res = cc.ansatz_resource_counts(5, 3, L=2)
print("Table III row ((5,3,3))_3:", {k: res[k] for k in ("E", "P_layer", "params", "XY_per_layer",
                                                           "MS_per_layer", "Z_per_layer", "MS_budget")})
record("Table III P_l", 90, res["P_layer"], "{:d}"); record("Table III params", 180, res["params"], "{:d}")
record("Table III MS budget", 40, res["MS_budget"], "{:d}")

seq = layer_gate_sequence(5, 3, cc.all_to_all_edges(5))
counts = {g: sum(1 for s in seq if s["gate"] == g) for g in ("XY", "MS", "Z")}
print("gates per layer:", counts, "| parameter slots used:", sum(len(s["param_slots"]) for s in seq))
params = np.asarray(central["params"])
print("\nfirst layer, first XY gate and first two MS gates with their optimized values:")
for g in [seq[0]] + [s for s in seq if s["gate"] == "MS"][:2]:
    vals = [f"{params[0, i]:+.4f}" for i in g["param_slots"]]
    names = {"XY": "(phi, alpha)", "MS": "(phi, theta)", "Z": "(theta,)"}[g["gate"]]
    print(f"  {g['gate']:2s} on qudits {g['qudits']} levels {g['levels']}  {names} = {vals}")

Table III row ((5,3,3))_3: {'E': 10, 'P_layer': 90, 'params': 180, 'XY_per_layer': 20, 'MS_per_layer': 20, 'Z_per_layer': 10, 'MS_budget': 40}
gates per layer: {'XY': 20, 'MS': 20, 'Z': 10} | parameter slots used: 90

first layer, first XY gate and first two MS gates with their optimized values:
  XY on qudits (0,) levels (0, 1)  (phi, alpha) = ['+1.6659', '+4.1475']
  MS on qudits (0, 1) levels (0, 1)  (phi, theta) = ['+4.9309', '+3.2533']
  MS on qudits (0, 1) levels (1, 2)  (phi, theta) = ['+3.1474', '+5.3345']


### (b) The $[[5,1,3]]_{Z_3}$ encoder of App. C: Fourier + CSUB + CZ, compiled to 36 MS gates

The codewords are graph states on the five-cycle under the label constraint. The circuit is
(i) Fourier gates on qudits 0–3, (ii) $\mathrm{CSUB}_{i\to 4}$ for $i = 0..3$ with the label on
qudit 4, (iii) one $\mathrm{CZ} = \exp[i\tfrac{2\pi}{3} N\otimes N]$ per cycle edge — nine
two-qudit Cliffords, each compiled exactly into **four MS gates** plus single-qudit gates.

**One CZ → 4 × MS, by hand.** With $N = \mathrm{diag}(0,1,2) = I - Z_{01} - Z_{12}$
($Z_{jk}$ the embedded $\sigma_z$ on levels $j,k$),
$N\otimes N = I\otimes I - (I\otimes Z_{01} + I\otimes Z_{12} + Z_{01}\otimes I + Z_{12}\otimes I)
+ \sum_{ab,cd\in\{01,12\}} Z_{ab}\otimes Z_{cd}$. The four cross terms are the four MS blocks:
$\exp[i\tfrac{2\pi}{3} Z_{ab}\otimes Z_{cd}] = \exp[-i\tfrac{\theta}{2} Z_{ab}\otimes Z_{cd}]$ with
$\theta = -4\pi/3$, and the MS gate at $\phi = 0$, $U_{\rm MS} = \exp[-i\tfrac{\theta}{4} S_\phi^2]$,
conjugated by subspace Hadamards ($H\sigma_x H = \sigma_z$) gives exactly this ZZ rotation times the
single-active phases $e^{-i\theta/4}$ that a diagonal correction removes; a level permutation
$W$ moves the second qudit's transition from $(a,b)$ to $(c,d)$. The single-qudit terms are one
diagonal gate per qudit and $I\otimes I$ is a global phase.

In [6]:
from src.stabilizer_encoder import (
    cz_gate, cz_gatelist, zz_from_ms, pair_gatelist_unitary, frobenius_distance_up_to_phase,
    subspace_hadamard, subspace_z, five_qudit_encoder_gatelist, count_ms, count_single,
    verify_encoder)
from src.gates import MS_gate

np.set_printoptions(precision=3, suppress=True, linewidth=120)
theta = -4 * np.pi / 3
print("theta =", theta, "= -4pi/3 ;  exp(-i theta/2 ZZ) = exp(+i 2pi/3 ZZ)\n")

# one block, Z_01 (x) Z_12, written out gate by gate
block = zz_from_ms(theta, (0, 1), (1, 2))
for g in block:
    if g[0] == "s":
        print(f"single-qudit gate on pair slot {g[1]}:\n{np.round(g[2], 3)}")
    else:
        print(f"MS gate: phi = {g[1]}, theta = {g[2]:.4f}, levels ({g[3]}, {g[4]})")
U_block = pair_gatelist_unitary(block)
target = np.diag(np.exp(-1j * theta / 2 * np.kron(np.diag(subspace_z(0, 1)), np.diag(subspace_z(1, 2)))))
print(f"\n|| block - exp(-i theta/2 Z_01 x Z_12) ||_F (up to global phase) = "
      f"{frobenius_distance_up_to_phase(U_block, target):.1e}")

# the four blocks + local diagonals = CZ
U_cz = pair_gatelist_unitary(cz_gatelist(1))
print(f"CZ from 4 MS blocks: {count_ms(cz_gatelist(1))} MS gates, "
      f"Frobenius residual {frobenius_distance_up_to_phase(U_cz, cz_gate()):.1e}")

theta = -4.1887902047863905 = -4pi/3 ;  exp(-i theta/2 ZZ) = exp(+i 2pi/3 ZZ)

single-qudit gate on pair slot 0:
[[ 0.707-0.j  0.707-0.j  0.   -0.j]
 [ 0.707-0.j -0.707-0.j  0.   -0.j]
 [ 0.   -0.j  0.   -0.j  1.   -0.j]]
single-qudit gate on pair slot 1:
[[ 0.   +0.j  0.707+0.j  0.707+0.j]
 [ 0.   +0.j  0.707+0.j -0.707+0.j]
 [ 1.   +0.j  0.   +0.j  0.   +0.j]]
MS gate: phi = 0.0, theta = -4.1888, levels (0, 1)
single-qudit gate on pair slot 0:
[[ 0.354-0.612j  0.354-0.612j  0.   +0.j   ]
 [ 0.354-0.612j -0.354+0.612j  0.   +0.j   ]
 [ 0.   +0.j     0.   +0.j     1.   +0.j   ]]
single-qudit gate on pair slot 1:
[[ 0.   +0.j     0.   +0.j     1.   +0.j   ]
 [ 0.354-0.612j  0.354-0.612j  0.   +0.j   ]
 [ 0.354-0.612j -0.354+0.612j  0.   +0.j   ]]

|| block - exp(-i theta/2 Z_01 x Z_12) ||_F (up to global phase) = 8.1e-16
CZ from 4 MS blocks: 4 MS gates, Frobenius residual 3.1e-15


In [7]:
gates = five_qudit_encoder_gatelist()
checks = verify_encoder()
print(f"full encoder: {count_ms(gates)} MS gates + {count_single(gates)} single-qudit gates "
      f"({checks['two_qudit_cliffords']} two-qudit Cliffords x 4 MS)")
print("CZ / CZ^dag / CSUB residuals :", f"{checks['cz_4ms_fro']:.1e}",
      f"{checks['czdag_4ms_fro']:.1e}", f"{checks['csub_4ms_fro']:.1e}")
print("codeword residuals (|0000a> -> |a_bar|, up to global phase):",
      [f"{x:.1e}" for x in checks["codeword_devs_up_to_phase"]])
record("App. C MS budget of [[5,1,3]] encoder", 36, checks["ms_budget"], "{:d}")
record("App. C codeword residual (<= 1e-14)", "<=1e-14", max(checks["codeword_devs_up_to_phase"]), "{:.1e}")

full encoder: 36 MS gates + 174 single-qudit gates (9 two-qudit Cliffords x 4 MS)
CZ / CZ^dag / CSUB residuals : 3.1e-15 5.1e-15 4.7e-15
codeword residuals (|0000a> -> |a_bar|, up to global phase): ['9.3e-15', '9.4e-15', '9.5e-15']


## 4. Noise model: one calibrated gate step (Eq. 8)

$\mathcal{N}_{\rm gate} = \mathcal{A}_{\sigma_a^2}\circ(\mathcal{D}^{(c)}\otimes\mathcal{D}^{(t)}\otimes\bigotimes_{\rm sp}\mathcal{D}^{(\rm sp)})$.
The dephasing factors are the Gaussian averages of the stochastic-phase model (App. B): level
$|j\rangle$ acquires phase $f(j)\Phi$, $\Phi\sim\mathcal N(0,\sigma_p^2)$, with coupling
$f(j) = j$ on spectators, $f = 0$ on the control level and $j$ elsewhere on the control qudit,
and $f = 1, 2, 3$ (control level, target level, other) on the target qudit. The Kraus family
$E_n = \sum_k \frac{(f_k\sigma_p)^n}{\sqrt{n!}}e^{-\sigma_p^2 f_k^2/2}|k\rangle\langle k|$
damps the coherence between levels with coupling difference $\Delta f$ by
$\sum_n E_n[i]\overline{E_n[j]} = e^{-\frac12\sigma_p^2\Delta f^2}$.

The angle-fluctuation term is exact dephasing in the eigenbasis of the MS generator
$S_\phi^2$ (Sec. III); its Kraus family is built by the same construction with the eigenvalue
tiers $g\in\{1, 1/4, 0\}$ as couplings.

The control-qudit map used here is the level-resolved one of App. B (couplings (0,1,2) at $q=3$, "B3"); the published uniform map of Meth et al. ((0,2,2), "B2") is available as `control_qudit_kraus_simplified` and is not used by any reported number. See `docs/NOISE_MODEL.md` §9 and `docs/HANDOFF.md` item B1.

In [8]:
from src.correlated_noise import (control_qudit_kraus, target_qudit_kraus, spectator_qudit_kraus,
                                  ms_jitter_kraus, verify_kraus_completeness,
                                  verify_matrix_kraus_completeness)

eta = cc.ETA_CAL
fams = {"control (level 0)": control_qudit_kraus(3, control_level=0, eta=eta, n_max=5),
        "target (levels 0,1)": target_qudit_kraus(3, control_level=0, target_level=1, eta=eta, n_max=5),
        "spectator": spectator_qudit_kraus(3, eta=eta, n_max=5)}
print("Sum_n E_n^+ E_n = I  (max deviation; the series is truncated at n_max):")
for label, diags in fams.items():
    _, dev5 = verify_kraus_completeness(diags)
    dev12 = verify_kraus_completeness({"control (level 0)": control_qudit_kraus(3, 0, eta, 12),
                                       "target (levels 0,1)": target_qudit_kraus(3, 0, 1, eta, 12),
                                       "spectator": spectator_qudit_kraus(3, eta, 12)}[label])[1]
    print(f"  {label:20s} n_max=5: {dev5:.1e}   n_max=12: {dev12:.1e}")
jit = ms_jitter_kraus(3, cc.SIGMA_A2, n_max=10)
print(f"  angle fluctuation    n_max=10: {verify_matrix_kraus_completeness(jit)[1]:.1e}  "
      f"({len(jit)} Kraus operators on the pair)")

# coherence multipliers of the spectator family vs the Gaussian formula
diags = spectator_qudit_kraus(3, eta=eta, n_max=12)
D = np.stack(diags)                       # (n_max+1, 3)
C = (D.T @ D.conj()).real                 # C[i,j] = sum_n E_n[i] conj(E_n[j])
expected = cc.coherence_multipliers(cc.SIGMA_P2, [0, 1, 2])
print("\nspectator coherence matrix C[i,j] = sum_n E_n[i] E_n[j]^* at eta = 0.9296:\n", np.round(C, 4))
print("expected e^{-sigma_p^2 Delta_f^2 / 2} for Delta_f = 0, 1, 2:", np.round(expected, 4),
      "  (paper: 1.0 / 0.9642 / 0.8642)")
record("coherence multiplier Delta_f=1", 0.9642, C[0, 1]); record("coherence multiplier Delta_f=2", 0.8642, C[0, 2])
record("Kraus completeness deviation (n_max=12)", 0.0, max(verify_kraus_completeness(d)[1] for d in
       (control_qudit_kraus(3, 0, eta, 12), target_qudit_kraus(3, 0, 1, eta, 12), diags)), "{:.1e}")

Sum_n E_n^+ E_n = I  (max deviation; the series is truncated at n_max):
  control (level 0)    n_max=5: 6.7e-07   n_max=12: 1.1e-16
  target (levels 0,1)  n_max=5: 6.4e-05   n_max=12: 3.7e-13
  spectator            n_max=5: 6.7e-07   n_max=12: 1.1e-16
  angle fluctuation    n_max=10: 7.8e-16  (11 Kraus operators on the pair)

spectator coherence matrix C[i,j] = sum_n E_n[i] E_n[j]^* at eta = 0.9296:
 [[1.    0.964 0.864]
 [0.964 1.    0.964]
 [0.864 0.964 1.   ]]
expected e^{-sigma_p^2 Delta_f^2 / 2} for Delta_f = 0, 1, 2: [1.    0.964 0.864]   (paper: 1.0 / 0.9642 / 0.8642)


**One noise round = 10 gate steps.** The exact engine (`src.decoders.exact_channel`) applies
each dephasing factor as an elementwise (Schur) product of the density matrix with the analytic
Gaussian coherence mask of that qudit's role, and the angle-fluctuation factor as a rotation into
the $S_\phi^2$ eigenbasis, a Schur product, and the rotation back. One round applies one step per
coupled pair of the all-to-all graph; the lower-indexed qudit of the pair is the control.

Note that at $q = 3$ with the driven levels $(0,1)$ the three role masks coincide: the control couplings $(0,1,2)$, the target couplings $(1,2,3)$ and the spectator couplings $(0,1,2)$ have the same pairwise differences, so the dephasing part of one gate step is the same on every qudit; the roles differ only through the angle-fluctuation factor on the driven pair (and, at $q \ge 4$, through the level-resolved couplings).

In [9]:
steps = cc.noise_round_steps(5, 3, s=1.0)
print(f"one noise round at s = 1: {len(steps)} gate steps (pairs):",
      [(st["qa"], st["qb"]) for st in steps])
st = steps[0]
print(f"\nstep 1 acts on pair (control {st['qa']}, target {st['qb']}); the three mask types:")
print("control mask  (f = 0,1,2):\n", np.round(st["deph"][st["qa"]].real, 4))
print("target mask   (f = 1,2,3):\n", np.round(st["deph"][st["qb"]].real, 4))
print("spectator mask(f = 0,1,2):\n", np.round(st["deph"][4].real, 4))
print("angle-fluctuation mask on the pair, eigenvalue tiers g in {1, 1/4, 0}: shape", st["jmask"].shape,
      "| distinct values", sorted(set(np.round(st["jmask"].real.ravel(), 4))))

one noise round at s = 1: 10 gate steps (pairs): [(0, 1), (0, 2), (0, 3), (0, 4), (1, 2), (1, 3), (1, 4), (2, 3), (2, 4), (3, 4)]

step 1 acts on pair (control 0, target 1); the three mask types:
control mask  (f = 0,1,2):
 [[1.    0.964 0.864]
 [0.964 1.    0.964]
 [0.864 0.964 1.   ]]
target mask   (f = 1,2,3):
 [[1.    0.964 0.864]
 [0.964 1.    0.964]
 [0.864 0.964 1.   ]]
spectator mask(f = 0,1,2):
 [[1.    0.964 0.864]
 [0.964 1.    0.964]
 [0.864 0.964 1.   ]]
angle-fluctuation mask on the pair, eigenvalue tiers g in {1, 1/4, 0}: shape (3, 3, 3, 3) | distinct values [np.float64(0.9768), np.float64(0.9869), np.float64(0.9985), np.float64(1.0)]


## 5. Recovery calculation (Table I)

For each code and strength we evolve the codeword blocks $\mathcal N(|\psi_i\rangle\langle\psi_j|)$
through one noise round and score

* **(a) the projective syndrome decoder** of App. F: the weight-$\le 2$ corrections of the
  completed channel basis, ordered by descending channel prior, claim orthogonalized syndrome
  sectors $\Pi_k$; outcome $k$ applies $E_k^\dagger$, the unclaimed remainder gets the identity;
* **(b) the Petz recovery** (Eq. F2) $\mathcal R(X) = P\mathcal N^\dagger(\mathcal N(P)^{-1/2}X\mathcal N(P)^{-1/2})P$
  with $\mathcal N$ and $\mathcal N^\dagger$ as exact superoperators;
* **(c) the certified optimal recovery**: the channel-fidelity SDP over the Choi matrix of the
  recovery, solved by a fixed-point iteration whose every iterate is a valid channel (lower bound)
  and whose dual candidate, shifted to feasibility, is an upper bound.

The figure of merit is the channel infidelity $1 - F_{\rm avg}$, $F_{\rm avg} = (KF_e + 1)/(K+1)$.

In [10]:
PAPER_T1 = {  # Table I, units of 1e-2: (central proj, central Petz, stab proj, stab Petz, ratio)
    0.125: (5.66, 0.70, 4.91, 0.23, 3.05), 0.25: (11.78, 2.05, 10.41, 0.84, 2.45),
    0.5: (23.17, 6.03, 21.05, 3.08, 1.96), 1.0: (39.62, 16.22, 37.16, 10.43, 1.56),
    1.5: (49.36, 26.32, 47.18, 19.42, 1.36)}
cells = {}
t0 = time.time()
print(f"{'s':>6s} | {'central proj':>12s} {'central Petz':>12s} | {'stab proj':>10s} {'stab Petz':>10s} | "
      f"{'ratio':>6s} || paper: proj/Petz/proj/Petz/ratio")
for s in cc.S_GRID:
    steps = cc.noise_round_steps(5, 3, s)
    cen = cc.channel_cell(CS_CEN, s, steps=steps)
    stab = cc.channel_cell(CS_STAB, s, steps=steps)
    cells[s] = (cen, stab)
    ratio = cen["petz_rate"] / stab["petz_rate"]
    print(f"{s:6.3f} | {100*cen['projective_rate']:12.2f} {100*cen['petz_rate']:12.2f} | "
          f"{100*stab['projective_rate']:10.2f} {100*stab['petz_rate']:10.2f} | {ratio:6.2f} || "
          f"{PAPER_T1[s][0]:.2f}/{PAPER_T1[s][1]:.2f}/{PAPER_T1[s][2]:.2f}/{PAPER_T1[s][3]:.2f}/{PAPER_T1[s][4]:.2f}")
    record(f"Table I s={s} central proj (1e-2)", PAPER_T1[s][0], 100 * cen["projective_rate"], "{:.2f}")
    record(f"Table I s={s} central Petz (1e-2)", PAPER_T1[s][1], 100 * cen["petz_rate"], "{:.2f}")
    record(f"Table I s={s} stabilizer proj (1e-2)", PAPER_T1[s][2], 100 * stab["projective_rate"], "{:.2f}")
    record(f"Table I s={s} stabilizer Petz (1e-2)", PAPER_T1[s][3], 100 * stab["petz_rate"], "{:.2f}")
    record(f"Table I s={s} Petz ratio", PAPER_T1[s][4], ratio, "{:.2f}")
print(f"({time.time()-t0:.0f} s)")

     s | central proj central Petz |  stab proj  stab Petz |  ratio || paper: proj/Petz/proj/Petz/ratio


 0.125 |         5.66         0.70 |       4.91       0.23 |   3.05 || 5.66/0.70/4.91/0.23/3.05


 0.250 |        11.78         2.05 |      10.41       0.84 |   2.45 || 11.78/2.05/10.41/0.84/2.45


 0.500 |        23.17         6.03 |      21.05       3.08 |   1.96 || 23.17/6.03/21.05/3.08/1.96


 1.000 |        39.62        16.22 |      37.16      10.43 |   1.56 || 39.62/16.22/37.16/10.43/1.56


 1.500 |        49.36        26.32 |      47.18      19.42 |   1.36 || 49.36/26.32/47.18/19.42/1.36
(10 s)


In [11]:
cen, stab = cells[1.0]
print("projective syndrome decoder sectors at the calibrated point (App. F):")
print(f"  central   : {cen['n_sectors']} sectors, dims {cen['sector_dim_counts']}, "
      f"claimed {cen['claimed_dim']} of 243, unclaimed {cen['rest_dim']}")
print(f"  stabilizer: {stab['n_sectors']} sectors, dims {stab['sector_dim_counts']} "
      f"(three-/two-/one-dimensional), claimed {stab['claimed_dim']} of 243")
cen05, stab05 = cells[0.5]
print(f"  stabilizer at s = 0.5: {stab05['n_sectors']} sectors (paper: 92 against 88 at s = 1)")
record("App. F sectors, central (s=1)", 81, cen["n_sectors"], "{:d}")
record("App. F stabilizer 3-dim sectors (s=1)", 74, stab["sector_dim_counts"].get(3, 0), "{:d}")
record("App. F stabilizer 2-dim sectors (s=1)", 7, stab["sector_dim_counts"].get(2, 0), "{:d}")
record("App. F stabilizer 1-dim sectors (s=1)", 7, stab["sector_dim_counts"].get(1, 0), "{:d}")
record("App. F stabilizer sectors (s=0.5)", 92, stab05["n_sectors"], "{:d}")

# the sigma_a^2 = 0 check of the Table I caption
steps0 = cc.noise_round_steps(5, 3, 1.0, with_angle_fluctuation=False)
cen0 = cc.channel_cell(CS_CEN, 1.0, steps=steps0, with_angle_fluctuation=False)
stab0 = cc.channel_cell(CS_STAB, 1.0, steps=steps0, with_angle_fluctuation=False)
print(f"\nsigma_a^2 = 0 at s = 1: Petz central {100*cen0['petz_rate']:.1f}, stabilizer {100*stab0['petz_rate']:.1f} "
      f"(paper 11.8 / 5.9); the term costs {100*(cen['petz_rate']/cen0['petz_rate']-1):.0f}% and "
      f"{100*(stab['petz_rate']/stab0['petz_rate']-1):.0f}% (paper 37% / 76%)")
record("Table I caption sigma_a^2=0 Petz central (1e-2)", 11.8, 100 * cen0["petz_rate"], "{:.1f}")
record("Table I caption sigma_a^2=0 Petz stabilizer (1e-2)", 5.9, 100 * stab0["petz_rate"], "{:.1f}")

projective syndrome decoder sectors at the calibrated point (App. F):
  central   : 81 sectors, dims {3: 81}, claimed 243 of 243, unclaimed 0
  stabilizer: 88 sectors, dims {1: 7, 2: 7, 3: 74} (three-/two-/one-dimensional), claimed 243 of 243
  stabilizer at s = 0.5: 92 sectors (paper: 92 against 88 at s = 1)



sigma_a^2 = 0 at s = 1: Petz central 11.8, stabilizer 5.9 (paper 11.8 / 5.9); the term costs 37% and 76% (paper 37% / 76%)


**(c) Certified optimal recovery.** The committed record behind App. F covers $s\in\{0.25, 0.5\}$
(Petz-over-optimal overheads 1.42–1.46 for the stabilizer code, 1.28–1.36 for the variational
codes); here the solver runs on every strength of `S_OPTIMAL`.

In [12]:
t0 = time.time()
opt = {}
print(f"{'s':>6s} | {'central opt [lb, ub]':>26s} {'Petz/opt':>9s} | {'stab opt [lb, ub]':>26s} {'Petz/opt':>9s} | ratio")
for s in S_OPTIMAL:
    steps = cc.noise_round_steps(5, 3, s)
    cen = cc.channel_cell(CS_CEN, s, steps=steps, optimal=True)
    stab = cc.channel_cell(CS_STAB, s, steps=steps, optimal=True)
    opt[s] = (cen, stab)
    assert cen["certified_opt_ge_petz"] and stab["certified_opt_ge_petz"]
    print(f"{s:6.3f} | [{100*cen['optimal_rate_lb']:9.4f}, {100*cen['optimal_rate_ub']:9.4f}] "
          f"{cen['petz_rate']/cen['optimal_rate_ub']:9.3f} | "
          f"[{100*stab['optimal_rate_lb']:9.4f}, {100*stab['optimal_rate_ub']:9.4f}] "
          f"{stab['petz_rate']/stab['optimal_rate_ub']:9.3f} | "
          f"{cen['optimal_rate_lb']/stab['optimal_rate_ub']:.2f}  "
          f"({cen['optimal_iters']}+{stab['optimal_iters']} it)")
print(f"({time.time()-t0:.0f} s)  rates in units of 1e-2; 'ratio' = certified lower bound of central / upper bound of stabilizer")
if 0.5 in opt:
    record("App. F Petz/optimal overhead, stabilizer (s=0.5)", 1.46,
           opt[0.5][1]["petz_rate"] / opt[0.5][1]["optimal_rate_ub"], "{:.2f}")
    record("App. F Petz/optimal overhead, central (s=0.5)", 1.34,
           opt[0.5][0]["petz_rate"] / opt[0.5][0]["optimal_rate_ub"], "{:.2f}")
if 0.25 in opt:
    record("App. F Petz/optimal overhead, stabilizer (s=0.25)", 1.42,
           opt[0.25][1]["petz_rate"] / opt[0.25][1]["optimal_rate_ub"], "{:.2f}")

     s |       central opt [lb, ub]  Petz/opt |          stab opt [lb, ub]  Petz/opt | ratio


 0.125 | [   0.5424,    0.5424]     1.291 | [   0.1654,    0.1655]     1.389 | 3.28  (20+20 it)


 0.250 | [   1.5579,    1.5580]     1.318 | [   0.5881,    0.5882]     1.424 | 2.65  (30+30 it)


 0.500 | [   4.5048,    4.5050]     1.339 | [   2.1160,    2.1161]     1.456 | 2.13  (60+40 it)


 1.000 | [  12.0439,   12.0440]     1.347 | [   7.0954,    7.0955]     1.470 | 1.70  (90+160 it)


 1.500 | [  19.7634,   19.7635]     1.332 | [  13.4009,   13.4011]     1.449 | 1.47  (130+490 it)
(147 s)  rates in units of 1e-2; 'ratio' = certified lower bound of central / upper bound of stabilizer


## 6. i.i.d. certification benchmark (Sec. V A, V C)

Each qudit independently suffers, with probability $p = 0.05$, one operator drawn uniformly from
the channel error basis; the lookup decoder tries the identity and every single-qudit inverse and
keeps the candidate closest to the code space; a shot fails when the recovered fidelity is below
$1/2$. Paper: $3.42\times10^{-4}$ vs $8.49\times10^{-4}$ ($2.48\times$) at $2\times10^6$ shots each.
With `N_SHOTS = 2_000_000` and `SEED_IID = 3000` this cell reproduces the paper's measurement
shot for shot (same random stream); a smaller `N_SHOTS` lands inside the Wilson intervals.

In [13]:
t0 = time.time()
iid = {}
for label, cs in (("central", CS_CEN), ("stabilizer", CS_STAB)):
    iid[label] = cc.iid_logical_error_rate(cs, p=cc.P_BENCH, n_shots=N_SHOTS, seed=SEED_IID)
    r = iid[label]
    print(f"{label:10s} LER = {r['ler']:.3e}  ({r['k']} errors / {r['n_shots']} shots), "
          f"Wilson 95% [{r['wilson95'][0]:.2e}, {r['wilson95'][1]:.2e}]")
ratio_iid = iid["stabilizer"]["ler"] / iid["central"]["ler"]
print(f"ratio stabilizer / central = {ratio_iid:.2f}   (paper 2.48)   [{time.time()-t0:.0f} s]")
record("i.i.d. LER central, p=0.05", 3.42e-4, iid["central"]["ler"], "{:.3e}")
record("i.i.d. LER stabilizer, p=0.05", 8.49e-4, iid["stabilizer"]["ler"], "{:.3e}")
record("i.i.d. ratio", 2.48, ratio_iid, "{:.2f}")

central    LER = 3.420e-04  (684 errors / 2000000 shots), Wilson 95% [3.17e-04, 3.69e-04]


stabilizer LER = 8.490e-04  (1698 errors / 2000000 shots), Wilson 95% [8.10e-04, 8.90e-04]
ratio stabilizer / central = 2.48   (paper 2.48)   [123 s]


**Weight-conditioned probes.** Exactly $w$ qudits err per shot. $\mathrm{LER}|_{w=1} = 0$ certifies
weight-1 correction (both codes); $\mathrm{LER}|_{w=2} = 0$ for both; the entire i.i.d. gap at
$p = 0.05$ traces to the weight-3 response, 0.289 (central) vs 0.729 (stabilizer). The probe shot
counts and seeds are those of the central code's selection run.

In [14]:
t0 = time.time()
probes = {}
for label, cs in (("central", CS_CEN), ("stabilizer", CS_STAB)):
    probes[label] = {}
    for w, shots, seed in ((1, 200_000, 1001), (2, 100_000, 1002), (3, 100_000, 1003)):
        r = cc.weight_conditioned_probe(cs, w, n_shots=shots, seed=seed)
        probes[label][w] = r
        print(f"{label:10s} LER|w={w} = {r['ler']:.4f}  ({r['k']} / {shots}; Wilson UB {r['wilson95'][1]:.1e}; "
              f"mean fidelity {r['mean_fidelity']:.6f})")
print(f"[{time.time()-t0:.0f} s]")
record("LER|w=1 central", 0.0, probes["central"][1]["ler"], "{:.0e}")
record("LER|w=1 stabilizer", 0.0, probes["stabilizer"][1]["ler"], "{:.0e}")
record("LER|w=2 central", 0.0, probes["central"][2]["ler"], "{:.0e}")
record("LER|w=3 central", 0.289, probes["central"][3]["ler"], "{:.3f}")
record("LER|w=3 stabilizer", 0.729, probes["stabilizer"][3]["ler"], "{:.3f}")

central    LER|w=1 = 0.0000  (0 / 200000; Wilson UB 1.9e-05; mean fidelity 1.000000)


central    LER|w=2 = 0.0000  (0 / 100000; Wilson UB 3.8e-05; mean fidelity 0.993202)


central    LER|w=3 = 0.2892  (28916 / 100000; Wilson UB 2.9e-01; mean fidelity 0.652490)


stabilizer LER|w=1 = 0.0000  (0 / 200000; Wilson UB 1.9e-05; mean fidelity 1.000000)


stabilizer LER|w=2 = 0.0000  (0 / 100000; Wilson UB 3.8e-05; mean fidelity 1.000000)


stabilizer LER|w=3 = 0.7287  (72873 / 100000; Wilson UB 7.3e-01; mean fidelity 0.346501)
[47 s]


## 7. Summary: paper value vs notebook value

In [15]:
def _fmt(v, fmt):
    try:
        return fmt.format(v)
    except (ValueError, TypeError):
        return str(v)

w = max(len(r[0]) for r in SUMMARY)
print(f"{'quantity':{w}s} | {'paper':>12s} | {'notebook':>12s}")
print("-" * (w + 31))
for name, paper, value, fmt in SUMMARY:
    print(f"{name:{w}s} | {_fmt(paper, fmt):>12s} | {_fmt(value, fmt):>12s}")
print(f"\ntotal wall time: {(time.time() - T_START)/60:.1f} min")

quantity                                           |        paper |     notebook
---------------------------------------------------------------------------------
Table IV residual, central (full-basis loss)       |         4.60 |         4.60
Table IV residual, stabilizer                      |       ~1e-30 |      2.6e-30
Table II central A1                                |        2.441 |        2.441
Table II central A2                                |        2.423 |        2.423
Table II central B1                                |        2.463 |        2.463
Table II central B2                                |        3.123 |        3.123
Table II stabilizer A1 (=B1)                       |        2.222 |        2.222
Table II stabilizer A2 (=B2)                       |        1.975 |        1.975
Table II stabilizer B2-A2                          |      0.0e+00 |      8.0e-15
Table III P_l                                      |           90 |           90
Table III params           